In [13]:
import pandas as pd

# Load the dataset
df = pd.read_csv('analysis.csv')

# Preview the first 5 rows
print(df.head())

# Check the dataset info: data types, non-null counts
print(df.info())

# Check the shape (rows, columns)
print(df.shape)

   userId  movieId  rating            timestamp                 title  \
0       1        1     4.0  2000-07-30 18:45:03             Toy Story   
1       1        3     4.0  2000-07-30 18:20:47      Grumpier Old Men   
2       1        6     4.0  2000-07-30 18:37:04                  Heat   
3       1       47     5.0  2000-07-30 19:03:35  Seven (a.k.a. Se7en)   
4       1       50     5.0  2000-07-30 18:48:51        Usual Suspects   

                                        genres  imdbId  tmdbId    year  
0  Adventure|Animation|Children|Comedy|Fantasy  114709     862  1995.0  
1                               Comedy|Romance  113228   15602  1995.0  
2                        Action|Crime|Thriller  113277     949  1995.0  
3                             Mystery|Thriller  114369     807  1995.0  
4                       Crime|Mystery|Thriller  114814     629  1995.0  
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 9 columns):
 #   Column     Non-Null

In [14]:
# Count missing values in each column
print(df.isnull().sum())

userId        0
movieId       0
rating        0
timestamp     0
title         0
genres        0
imdbId        0
tmdbId        0
year         18
dtype: int64


In [15]:
# Drop rows with missing values (if any)
df = df.dropna()

# Or, for specific columns, e.g., 'rating', you could do:
# df = df[df['rating'].notnull()]

In [4]:
# Check for duplicate rows
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Remove duplicates
df = df.drop_duplicates()

Number of duplicate rows: 0


In [16]:
# Ensure 'rating' is numeric
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Check again for missing values after coercion
print(df['rating'].isnull().sum())

0


In [17]:
# Group by movie title
movie_stats = df.groupby('title').agg(
    average_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

# Preview
print(movie_stats.head())

                              title  average_rating  num_ratings
0                               '71             4.0            1
1  'Hellboy': The Seeds of Creation             4.0            1
2                   'Round Midnight             3.5            2
3                      'Salem's Lot             5.0            1
4                'Til There Was You             4.0            2


In [7]:
# Filter movies with more than 50 ratings
popular_movies = movie_stats[movie_stats['num_ratings'] > 50]

# Merge with the original dataframe to keep only popular movies
df_filtered = df[df['title'].isin(popular_movies['title'])]

print(df_filtered.shape)

(41455, 9)


In [8]:
# Create user-movie matrix
user_movie_matrix = df_filtered.pivot_table(
    index='userId', 
    columns='title', 
    values='rating'
)

# Preview the matrix
print(user_movie_matrix.head())

title   10 Things I Hate About You  12 Angry Men  2001: A Space Odyssey  \
userId                                                                    
1                              NaN           NaN                    NaN   
2                              NaN           NaN                    NaN   
3                              NaN           NaN                    NaN   
4                              NaN           5.0                    NaN   
5                              NaN           NaN                    NaN   

title   28 Days Later  300  40-Year-Old Virgin  A.I. Artificial Intelligence  \
userId                                                                         
1                 NaN  NaN                 NaN                           NaN   
2                 NaN  NaN                 NaN                           NaN   
3                 NaN  NaN                 NaN                           NaN   
4                 NaN  NaN                 NaN                           N

In [9]:
# Check shape
print(f"Matrix shape: {user_movie_matrix.shape}")

# Preview some rows and columns
print(user_movie_matrix.iloc[:5, :5])

Matrix shape: (607, 445)
title   10 Things I Hate About You  12 Angry Men  2001: A Space Odyssey  \
userId                                                                    
1                              NaN           NaN                    NaN   
2                              NaN           NaN                    NaN   
3                              NaN           NaN                    NaN   
4                              NaN           5.0                    NaN   
5                              NaN           NaN                    NaN   

title   28 Days Later  300  
userId                      
1                 NaN  NaN  
2                 NaN  NaN  
3                 NaN  NaN  
4                 NaN  NaN  
5                 NaN  NaN  


In [10]:
# Replace NaN with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Verify
print(user_movie_matrix_filled.head())

title   10 Things I Hate About You  12 Angry Men  2001: A Space Odyssey  \
userId                                                                    
1                              0.0           0.0                    0.0   
2                              0.0           0.0                    0.0   
3                              0.0           0.0                    0.0   
4                              0.0           5.0                    0.0   
5                              0.0           0.0                    0.0   

title   28 Days Later  300  40-Year-Old Virgin  A.I. Artificial Intelligence  \
userId                                                                         
1                 0.0  0.0                 0.0                           0.0   
2                 0.0  0.0                 0.0                           0.0   
3                 0.0  0.0                 0.0                           0.0   
4                 0.0  0.0                 0.0                           0

In [18]:
# Merge the filtered movies with movie statistics
filtered_movie_stats = df_filtered.merge(
    movie_stats,       # dataframe with average_rating and num_ratings
    on='title',        # merge on movie title
    how='left'         # keep all filtered movies
)

# Preview the combined dataframe
print(filtered_movie_stats.head())

   userId  movieId  rating            timestamp                 title  \
0       1        1     4.0  2000-07-30 18:45:03             Toy Story   
1       1        3     4.0  2000-07-30 18:20:47      Grumpier Old Men   
2       1        6     4.0  2000-07-30 18:37:04                  Heat   
3       1       47     5.0  2000-07-30 19:03:35  Seven (a.k.a. Se7en)   
4       1       50     5.0  2000-07-30 18:48:51        Usual Suspects   

                                        genres  imdbId  tmdbId    year  \
0  Adventure|Animation|Children|Comedy|Fantasy  114709     862  1995.0   
1                               Comedy|Romance  113228   15602  1995.0   
2                        Action|Crime|Thriller  113277     949  1995.0   
3                             Mystery|Thriller  114369     807  1995.0   
4                       Crime|Mystery|Thriller  114814     629  1995.0   

   average_rating  num_ratings  
0        3.920930          215  
1        3.259615           52  
2        3.916667

In [19]:
# Save the combined filtered movies with stats
filtered_movie_stats.to_csv('filtered_movie_stats.csv', index=False)

print("Filtered movie stats saved as 'filtered_movie_stats.csv'")

Filtered movie stats saved as 'filtered_movie_stats.csv'


In [20]:
# Round average_rating to 2 decimal places
filtered_movie_stats['average_rating'] = filtered_movie_stats['average_rating'].round(2)

# Preview
print(filtered_movie_stats[['title', 'average_rating', 'num_ratings']].head())

                  title  average_rating  num_ratings
0             Toy Story            3.92          215
1      Grumpier Old Men            3.26           52
2                  Heat            3.92          108
3  Seven (a.k.a. Se7en)            3.98          203
4        Usual Suspects            4.24          204


In [21]:
# Save the cleaned CSV with rounded average_rating
filtered_movie_stats.to_csv('filtered_movie_stats.csv', index=False)

print("Filtered movie stats saved with average_rating rounded to 2 decimals.")

Filtered movie stats saved with average_rating rounded to 2 decimals.
